In [ ]:
## Step 2: MSSR Batch Processing (Terminal)

Due to the memory-intensive nature of the MSSR algorithm, this step is provided as a standalone Python script designed to be executed from the Conda terminal to prevent kernel crashes.

**Execution Instructions:**

1. Open the Anaconda Prompt.
2. Activate your conda environment (if applicable):
   `conda activate your_env_name`
3. Navigate to the directory containing the script:
   `cd "path\to\your\script"`
4. Run the Python file adjusting the paramets as needed:
   `python 02_run_mssr_batch.py`

The script will automatically process all `.tif` files in the input directory and save the super-resolved images with the `_MSSR.tif` suffix in the output folder.

Script:
    
import os
import glob
import tifffile
import numpy as np
from mssr_core import sfMSSR

print("====================================================")
print("  STARTING MSSR BATCH PROCESSING  ")
print("====================================================\n")

# --- 1. FOLDER CONFIGURATION ---
# Replace the strings with the actual path to your images
# The "r" prefix is important in Windows to read backslashes correctly
input_dir = r"input_dir"

# Results will be saved here. If it doesn't exist, Python will create it.
output_dir = r"output_dir"

os.makedirs(output_dir, exist_ok=True)

# --- 2. MSSR PARAMETERS ---
AMP = 5
FWHM = 3.8
ORDER = 0
MESH = True
FTI = False        # False = Bicubic Interpolation
INT_NORM = True    # True = Active intensity normalization

# Search for images in the input directory
search_path = os.path.join(input_dir, "*.tif")
image_files = [f for f in glob.glob(search_path) if "_MSSR" not in f]

print(f"Searching in: {input_dir}")
print(f"Found {len(image_files)} images to process.\n")

# --- 3. PROCESSING LOOP ---
for full_path in image_files:
    # Extract only the filename from the full path
    filename = os.path.basename(full_path)
    print(f"Processing: {filename} ...")
    
    # Read the raw image
    image = tifffile.imread(full_path)
    
    # Apply the MSSR algorithm
    result = sfMSSR(
        img=image, 
        fwhm=FWHM, 
        amp=AMP, 
        order=ORDER, 
        mesh=MESH, 
        ftI=FTI, 
        intNorm=INT_NORM
    )
    
    # Create the output filename and its new path
    output_filename = filename.replace(".tif", "_MSSR.tif")
    save_path = os.path.join(output_dir, output_filename)
    
    # Save in float32 format optimized for Fiji
    tifffile.imwrite(save_path, result.astype(np.float32), imagej=True)
    print(f"   [OK] Saved to output folder.\n")

print("====================================================")
print("PROCESS COMPLETED! Check your output folder.")
print("====================================================")